In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

np.random.seed(42)

# Варіант 5 — Дніпро
base_temp = 10.0
amplitude = 14
city = "Дніпро"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []

for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):

        seasonal = amplitude * np.cos(
            (month - 7) / 12 * 2 * np.pi
        )

        noise = np.random.normal(0, 1.0)

        temp = round(
            base_temp + seasonal + noise, 1
        )

        diff = temp - base_temp

        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"

        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": temp,
            "сезон": season(month),
            "відхилення_від_норми": norm_cat
        })

climate = pd.DataFrame(rows)

climate

,місто,рік,місяць,температура,сезон,відхилення_від_норми
0,Дніпро,2021,1,-3.5,зима,холодніше
1,Дніпро,2021,2,-2.3,зима,холодніше
2,Дніпро,2021,3,3.6,весна,холодніше
3,Дніпро,2021,4,11.5,весна,звичайно
4,Дніпро,2021,5,16.8,весна,тепліше
5,Дніпро,2021,6,21.9,літо,тепліше
6,Дніпро,2021,7,25.6,літо,тепліше
7,Дніпро,2021,8,22.9,літо,тепліше
8,Дніпро,2021,9,16.5,осінь,тепліше
9,Дніпро,2021,10,10.5,осінь,звичайно


In [2]:
table = pd.crosstab(
    climate["сезон"],
    climate["відхилення_від_норми"]
)

table

відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,4,4,4
зима,0,0,12
літо,0,12,0
осінь,4,4,4


In [3]:
table_norm = pd.crosstab(
    climate["сезон"],
    climate["відхилення_від_норми"],
    normalize="index"
)

table_norm

відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,0.333333,0.333333,0.333333
зима,0.000000,0.000000,1.000000
літо,0.000000,1.000000,0.000000
осінь,0.333333,0.333333,0.333333


Найчастіші комбінації — "зима + холодніше" та
"літо + тепліше": кожна трапляється по 12 разів.

Це відповідає логіці побудови набору даних, оскільки
температура формується за сезонним коливанням:
узимку вона значно нижча за середньорічну, а влітку —
значно вища.

У нормалізованій таблиці замість кількостей показано
частки всередині кожного сезону.

Тому сума значень кожного рядка дорівнює 1.

Це зручно для порівняння сезонів між собою, оскільки
ми порівнюємо не абсолютні кількості, а структуру
категорій усередині кожного сезону.

In [4]:
chi2, p_value, dof, expected = chi2_contingency(table)

expected_df = pd.DataFrame(
    expected,
    index=table.index,
    columns=table.columns
)

print("χ² =", chi2)
print("p-value =", p_value)
print("Ступені свободи =", dof)

print("\nОчікувані частоти:")
print(expected_df)

χ² = 38.400000000000006
p-value = 9.381704108243484e-07
Ступені свободи = 6

Очікувані частоти:
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      5.0        5.0
зима                       2.0      5.0        5.0
літо                       2.0      5.0        5.0
осінь                      2.0      5.0        5.0


H₀: сезон і відхилення від норми незалежні.
H₁: між ними існує статистичний зв'язок.

Отримано:

χ² = 38.4
p-value ≈ 9.38 × 10⁻⁷.

Оскільки p-value < 0.05, нульову гіпотезу H₀
відхиляємо.

Отже, у цьому наборі даних сезон і відхилення
температури від норми статистично пов'язані.

Це очікуваний результат, оскільки обидві змінні
формуються на основі сезонної зміни температури.

In [5]:
print(expected_df)

print(
    "\nЧи всі expected >= 5?",
    (expected_df >= 5).all().all()
)

відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      5.0        5.0
зима                       2.0      5.0        5.0
літо                       2.0      5.0        5.0
осінь                      2.0      5.0        5.0

Чи всі expected >= 5? False


In [6]:
climate["категорія_2"] = climate[
    "відхилення_від_норми"
].replace({
    "холодніше": "не тепліше",
    "звичайно": "не тепліше"
})

table2 = pd.crosstab(
    climate["сезон"],
    climate["категорія_2"]
)

table2

категорія_2,не тепліше,тепліше
сезон,,
весна,8,4
зима,12,0
літо,0,12
осінь,8,4


In [7]:
chi2_2, p_2, dof_2, expected_2 = chi2_contingency(table2)

expected2_df = pd.DataFrame(
    expected_2,
    index=table2.index,
    columns=table2.columns
)

print("χ² =", chi2_2)
print("p-value =", p_2)
print("dof =", dof_2)

print("\nОчікувані частоти:")
print(expected2_df)

χ² = 26.057142857142857
p-value = 9.278240763009508e-06
dof = 3

Очікувані частоти:
категорія_2  не тепліше  тепліше
сезон                           
весна               7.0      5.0
зима                7.0      5.0
літо                7.0      5.0
осінь               7.0      5.0


У початковій таблиці умова застосовності χ² не
виконувалась, оскільки для категорії "звичайно"
очікувані частоти становили лише 2.

Тому категорії "холодніше" і "звичайно" було
об'єднано в категорію "не тепліше".

Після об'єднання всі очікувані частоти становлять
5 або 7, тобто умова expected >= 5 виконується.

Повторний тест дав:

χ² ≈ 26.06
p-value ≈ 9.28 × 10⁻⁶.

Оскільки p < 0.05, H₀ знову відхиляємо.

Отже, навіть після коректного об'єднання категорій
висновок не змінився: сезон і відхилення температури
від норми статистично пов'язані.

In [8]:
season_counts = climate["сезон"].value_counts()

season_counts

,count
сезон,
зима,12
весна,12
літо,12
осінь,12


In [9]:
season_counts = season_counts.reindex([
    "зима", "весна", "літо", "осінь"
])

expected_seasons = np.array([
    12, 12, 12, 12
])

chi2_season, p_season = chisquare(
    f_obs=season_counts,
    f_exp=expected_seasons
)

print("χ² =", chi2_season)
print("p-value =", p_season)

χ² = 0.0
p-value = 1.0


H₀: чотири сезони представлені в наборі однаково,
тобто кожен має частку 25%.

Фактично кожен сезон трапляється рівно 12 разів із 48,
тобто також становить 25%.

Отримано:

χ² = 0
p-value = 1.

Немає підстав відхиляти H₀.

Результат повністю очікуваний, оскільки набір містить
усі 12 місяців кожного з чотирьох років. Кожен сезон
складається з трьох місяців, тому кожен представлений
рівно 12 разів.

H₀: розподіл категорій відхилення температури від норми
становить:

"холодніше" — 40%;
"звичайно" — 20%;
"тепліше" — 40%.

Таке припущення є обґрунтованим, оскільки категорія
"звичайно" охоплює лише температури в межах ±3 °C
від середньорічної температури.

Водночас сезонна амплітуда Дніпра становить 14 °C,
тому значна частина зимових і літніх місяців має
потрапляти до категорій "холодніше" та "тепліше".

In [10]:
observed = climate[
    "відхилення_від_норми"
].value_counts().reindex([
    "холодніше",
    "звичайно",
    "тепліше"
])

print("Фактичні частоти:")
print(observed)

Фактичні частоти:
відхилення_від_норми
холодніше    20
звичайно      8
тепліше      20
Name: count, dtype: int64


In [11]:
n = len(climate)

expected_custom = np.array([
    0.40 * n,
    0.20 * n,
    0.40 * n
])

chi2_custom, p_custom = chisquare(
    f_obs=observed,
    f_exp=expected_custom
)

print("Очікувані частоти:", expected_custom)
print("χ² =", chi2_custom)
print("p-value =", p_custom)

Очікувані частоти: [19.2  9.6 19.2]
χ² = 0.3333333333333333
p-value = 0.8464817248906141


Отримано:

χ² ≈ 0.333
p-value ≈ 0.846.

Оскільки p-value > 0.05, немає підстав відхиляти H₀.

Отже, фактичний розподіл категорій не суперечить
заявленій пропорції:

40% "холодніше",
20% "звичайно",
40% "тепліше".

Велике p-value не доводить, що саме ця пропорція є
правильною. Воно лише означає, що отримані дані не дають
достатніх підстав відхилити таке припущення.

Контрольні питання

1. Чому в статистиці χ² використовується квадрат
відхилення і ділення на очікувану частоту?

Квадрат використовується для того, щоб додатні та
від'ємні відхилення фактичних частот від очікуваних
не компенсували одне одного.

Крім того, більші відхилення отримують більшу вагу.

Ділення на очікувану частоту потрібне для врахування
масштабу: однакова абсолютна різниця є значно важливішою
для клітинки з малою очікуваною частотою, ніж для
клітинки з дуже великою частотою.


2. Як обчислюється очікувана частота клітинки таблиці
спряженості?

Очікувана частота визначається як:

E = (сума рядка × сума стовпця) / загальна сума.

При припущенні незалежності ймовірність одночасного
потрапляння в певний рядок і стовпець дорівнює добутку
їхніх окремих ймовірностей.

Тому ця формула показує, скільки спостережень ми
очікували б у клітинці, якби дві категоріальні змінні
дійсно були незалежними.


3. Чим відрізняються твердження "велике p-value —
немає підстав відхилити H₀" та "доведено H₀"?

Велике p-value не доводить, що H₀ правильна.

Воно означає лише, що отримані дані не містять достатньо
сильних доказів проти H₀ при обраному рівні значущості.

Тому правильно говорити "не відхиляємо H₀", а не
"H₀ доведена".


4. Що робити, якщо expected < 5?

Якщо очікувана частота в деяких клітинках менша за 5,
наближення статистики критерію до χ²-розподілу може бути
недостатньо точним, а отриманий p-value — ненадійним.

Одним зі способів розв'язання проблеми є логічне
об'єднання близьких категорій.

У цій роботі категорії "холодніше" і "звичайно"
було об'єднано в "не тепліше".

Після цього всі очікувані частоти стали не меншими
за 5, і критерій було застосовано повторно.